# HeritageLens · GPU worker for Google Colab

This notebook turns a free Colab GPU into a **HeritageLens worker**. It *pulls* jobs from your HeritageLens
server over outbound HTTPS, so Colab needs no open ports, and posts the results back:

| Job | Model | What the archive gains |
|---|---|---|
| `ocr` | Tesseract 5 with `tessdata_best` Gujarati, Hindi, Marathi and Sanskrit models, or Qwen2.5-VL-3B | Searchable text for scanned letters, registers and survey sheets |
| `translate` | IndicTrans2 (AI4Bharat), falling back to NLLB-200 | English glosses for Gujarati, Hindi, Marathi and other Indic field notes |
| `caption` | BLIP-large | Descriptions that make uncaptioned photos findable |
| `reconstruct3d` | Depth Anything V2 relief, or COLMAP + gsplat Gaussian splatting | Explorable 3D digital twins of sites |

**Before you start**
1. *Runtime → Change runtime type →* **T4 GPU**. A100, H100 and P100 also work: bf16 is used on Ampere and newer, fp16 on older cards.
2. Start the HeritageLens API, then expose it with a quick tunnel (step 4).

Jobs are **leased** and kept alive by heartbeats. If Colab disconnects mid-job, the lease expires and the server
re-queues the job, so no work is lost. You can run several Colab sessions at once to add capacity.

## 1 · GPU

In [ ]:
import subprocess

import torch

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(smi.stdout.strip() or "nvidia-smi: no NVIDIA GPU visible")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    GPU_NAME = props.name.replace("NVIDIA ", "")
    print(f"torch {torch.__version__} · CUDA {torch.version.cuda} · {GPU_NAME} · "
          f"{props.total_memory / 2**30:.0f} GB · {'bf16' if props.major >= 8 else 'fp16'}")
else:
    GPU_NAME = "CPU"
    print("No GPU attached - Runtime > Change runtime type > T4 GPU (the worker still runs on CPU, slowly).")

## 2 · Tesseract and Indic OCR models

Ubuntu packages ship the *fast* models. The Indic ones are replaced with the LSTM **`tessdata_best`** models,
which are noticeably more accurate on old letterpress and typewritten pages.

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y tesseract-ocr tesseract-ocr-osd tesseract-ocr-guj tesseract-ocr-hin tesseract-ocr-mar tesseract-ocr-san > /dev/null

import re
import subprocess
import urllib.request
from pathlib import Path


def tesseract(*args: str) -> str:
    out = subprocess.run(["tesseract", *args], capture_output=True, text=True)
    return (out.stdout + out.stderr).strip()


TESSDATA = Path(re.search(r'"(.+?)"', tesseract("--list-langs")).group(1))
for lang in ("guj", "hin", "mar", "san", "eng"):
    url = f"https://github.com/tesseract-ocr/tessdata_best/raw/main/{lang}.traineddata"
    urllib.request.urlretrieve(url, TESSDATA / f"{lang}.traineddata")
print(tesseract("--version").splitlines()[0])
print("models:", ", ".join(tesseract("--list-langs").splitlines()[1:]))

## 3 · Install the worker

Installs the `heritage-gpu` package from your copy of the repository. Colab already ships torch and transformers,
so this takes about a minute.

* **IndicTrans2** (`IndicTransToolkit`) is optional. Without it, the worker translates with NLLB-200.
* **Multi-view twins** need `pycolmap` and `gsplat`. On first use, gsplat compiles its CUDA kernels; this takes
  5–10 minutes and happens once per Colab session. Without them, twins are single-photo depth reliefs.

In [ ]:
REPO_URL = "https://github.com/YOUR-USERNAME/HeritageLens"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
INSTALL_INDICTRANS = True  #@param {type:"boolean"}
INSTALL_MULTIVIEW = False  #@param {type:"boolean"}

import subprocess
import sys
from pathlib import Path

assert "YOUR-USERNAME" not in REPO_URL, "Set REPO_URL to your HeritageLens repository first."
SRC = Path("/content/HeritageLens")
if SRC.exists():
    subprocess.run(["git", "-C", str(SRC), "pull", "-q", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, str(SRC)], check=True)


def pip(*packages: str) -> bool:
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages]).returncode == 0


assert pip(f"{SRC / 'ml'}[models]"), "worker install failed - see the pip output above"
if INSTALL_INDICTRANS and not pip("IndicTransToolkit>=1.0.3"):
    print("IndicTransToolkit did not install: translations will use NLLB-200.")
if INSTALL_MULTIVIEW and not pip("pycolmap>=3.10", "gsplat>=1.4"):
    print("pycolmap/gsplat did not install: twins will be depth reliefs.")
!python -m heritage_gpu --version

## 4 · Connect to your HeritageLens server

On the machine running the API (from `backend/`: `python -m app.cli serve`, port 8000 by default), open a
free Cloudflare quick tunnel to that port. It needs no account:

```bash
cloudflared tunnel --url http://localhost:8000   # use your port if you started the API with --port
```

Copy the printed `https://<random>.trycloudflare.com` URL. Then, in Colab's **🔑 Secrets** panel, add these two
entries and enable notebook access:

| Secret | Value |
|---|---|
| `HERITAGE_API_URL` | the tunnel URL |
| `HERITAGE_TOKEN` | the server's `HL_WORKER_TOKEN` |

If the secrets are not set, you are prompted instead.

**Engine choice.** On a T4, Tesseract is fast and accurate on printed pages. `vlm` (Qwen2.5-VL-3B) copes better
with faded, skewed or handwritten scans but is roughly 10× slower, so prefer it on an A100 or H100.

In [ ]:
OCR_ENGINE = "tesseract"  #@param ["tesseract", "vlm"]
TRANSLATOR = "auto"  #@param ["auto", "indictrans2", "nllb"]
JOB_TYPES = "ocr,translate,caption,reconstruct3d"  #@param {type:"string"}
SPLAT_STEPS = 7000  #@param {type:"integer"}

import os
from getpass import getpass


def secret(name: str, prompt: str, hidden: bool = False) -> str:
    try:
        from google.colab import userdata

        value = userdata.get(name)
        if value:
            return value
    except Exception:  # not in Colab, secret missing, or notebook access not granted
        pass
    return (getpass if hidden else input)(prompt).strip()


os.environ.update(
    {
        "HERITAGE_API_URL": secret("HERITAGE_API_URL", "HeritageLens URL (https://...trycloudflare.com): "),
        "HERITAGE_TOKEN": secret("HERITAGE_TOKEN", "Worker token (HL_WORKER_TOKEN): ", hidden=True),
        "HERITAGE_NAME": f"Colab {globals().get('GPU_NAME', 'GPU')}",
        "HERITAGE_OCR_ENGINE": OCR_ENGINE,
        "HERITAGE_TRANSLATOR": TRANSLATOR,
        "HERITAGE_JOB_TYPES": JOB_TYPES,
        "HERITAGE_SPLAT_STEPS": str(SPLAT_STEPS),
        "HERITAGE_WORK_DIR": "/content/heritage-work",
    }
)
print(f"{os.environ['HERITAGE_NAME']} -> {os.environ['HERITAGE_API_URL']} ({JOB_TYPES})")

## 5 · Preflight check

Checks four things: that the tunnel answers, that the server accepts the token, the server's health and queue
depth, and the local accelerator. A non-zero exit means something must be fixed before running.

In [ ]:
!python -m heritage_gpu check

## 6 · Self-test (recommended once per session)

Loads every model and runs each engine on synthetic Gujarati, Hindi and English samples, without contacting the
server. The first run downloads about 5 GB of weights, which takes 3–5 minutes on Colab.

In [ ]:
!python -m heritage_gpu selftest

## 7 · Run the worker

Leases and processes jobs until you stop this cell (■). Stopping hands the current job back to the queue, so it
is picked up again rather than lost. Follow progress in the HeritageLens web app under **Archive → Pipeline**.

In [ ]:
!python -m heritage_gpu run --preload

---
### Demo without models

`--fake` swaps in deterministic, model-free engines. They speak the full protocol and label their output
*synthetic*, which is handy for rehearsing a demo or testing a deployment from any machine:

```bash
python -m heritage_gpu run --fake --idle-exit 60
```

### Troubleshooting

| Symptom | Fix |
|---|---|
| `token : REJECTED` | `HERITAGE_TOKEN` must equal the server's `HL_WORKER_TOKEN`. |
| `server unreachable`, or HTTP 502/530 | The tunnel or the laptop went to sleep. Restart `cloudflared` and update `HERITAGE_API_URL`. |
| `CUDA out of memory` | The job is retried with backoff. Lower `SPLAT_STEPS`, or use `TRANSLATOR = "nllb"` on a T4. |
| Session disconnects | Colab drops idle tabs after about 90 minutes. Leased jobs return to the queue automatically. Re-run steps 4 to 7. |
| Slow first twin | gsplat compiles its CUDA kernels once per session (5–10 minutes). |